In [ ]:
# ---------------------------------------------------------------------------
# CONFIGURATION CELL
# ---------------------------------------------------------------------------
# Set to True to load the Img2Img pipeline, False for the Txt2Img pipeline.
# This is the ONLY cell you need to change between your two notebook copies.

USE_IMG2IMG = False

In [ ]:
import os, sys

p = "/kaggle/input/sage-zrok-token/.zrok_api_key"
zrok_token = None

if os.path.isfile(p):
    with open(p, "r", encoding="utf-8", errors="ignore") as f:
        zrok_token = f.read().strip()

if not zrok_token:
    print("❌ Token not found or empty:", p)
    sys.exit(1)

In [ ]:
import os
import shutil

print("Setting up models...")

# --- Copy Counterfeit-XL v2.5 if not already present ---
source = "/kaggle/input/counterfeit-xl-v25-sdxl"
dest = "/kaggle/working/John6666/counterfeit-xl-v25-sdxl-spo"

if os.path.exists(dest):
    print(f"✓ Counterfeit-XL v2.5 already exists at {dest}, skipping copy")
else:
    print(f"  Copying Counterfeit-XL v2.5... (this may take a moment)")
    shutil.copytree(source, dest)
    print(f"  ✓ Copied to {dest}")

# --- Copy ControlNet OpenPose for SDXL if not already present ---
controlnet_source = "/kaggle/input/controlnet-openpose-sdxl"
controlnet_dest = "/kaggle/working/controlnet-openpose-sdxl"

if os.path.exists(controlnet_dest):
    print(f"✓ ControlNet OpenPose SDXL already exists at {controlnet_dest}, skipping copy")
else:
    print(f"  Copying ControlNet OpenPose SDXL...")
    shutil.copytree(controlnet_source, controlnet_dest)
    print(f"  ✓ Copied to {controlnet_dest}")

print("✅ All models ready!")

In [ ]:
import torch
from diffusers import (
    ControlNetModel,
    StableDiffusionXLControlNetPipeline,
    StableDiffusionXLControlNetImg2ImgPipeline,
    DPMSolverMultistepScheduler
)

# --- Device ---
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Model paths ---
local_model_path = "/kaggle/working/John6666/counterfeit-xl-v25-sdxl-spo"
local_controlnet_path = "/kaggle/working/controlnet-openpose-sdxl"

# --- Load ControlNet (common for both pipelines) ---
print("Loading ControlNet model...")
controlnet = ControlNetModel.from_pretrained(
    local_controlnet_path,
    torch_dtype=torch.float16
).to(device)
print("✓ ControlNet loaded.")

pipe = None

# --- Conditionally load the main pipeline based on the USE_IMG2IMG flag ---
if not USE_IMG2IMG:
    print("Mode: Txt2Img. Loading pipeline...")
    pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
        local_model_path,
        controlnet=controlnet,
        torch_dtype=torch.float16,
        variant="fp16"
    ).to(device)
    print("✓ Txt2Img pipeline with ControlNet loaded.")
else:
    print("Mode: Img2Img. Loading pipeline...")
    pipe = StableDiffusionXLControlNetImg2ImgPipeline.from_pretrained(
        local_model_path,
        controlnet=controlnet,
        torch_dtype=torch.float16,
        variant="fp16"
    ).to(device)
    print("✓ Img2Img pipeline with ControlNet loaded.")

# --- Configure Scheduler and Safety Checker ---
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config,
    algorithm_type="sde-dpmsolver++"
)
pipe.safety_checker = lambda images, clip_input=None: (images, [False] * len(images))

In [ ]:
from PIL import Image
import torch
from typing import Optional
import random

def generate_txt2img_xl(
    pipe, # StableDiffusionXLControlNetPipeline
    prompt: str,
    negative_prompt: Optional[str] = None,
    height: int = 1024,
    width: int = 1024,
    num_inference_steps: int = 40,
    guidance_scale: float = 7.5,
    seed: Optional[int] = None,
    control_image: Optional[Image.Image] = None,
    controlnet_scale: float = 0.5,
):
    """Generates an image from text using an SDXL pipeline, with optional ControlNet."""
    if seed is None:
        seed = random.randint(0, 2**32 - 1)
    generator = torch.Generator(device="cuda").manual_seed(seed)

    use_controlnet = control_image is not None
    pipeline_image = control_image if use_controlnet else Image.new("RGB", (width, height), (0, 0, 0))

    output = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=pipeline_image, # This is the control image for this pipeline
        height=height,
        width=width,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        controlnet_conditioning_scale=controlnet_scale if use_controlnet else 0.0,
        generator=generator,
    )
    return output.images[0]

In [ ]:
from PIL import Image
import torch
from typing import Optional
import random

def generate_img2img_xl(
    pipe, # StableDiffusionXLControlNetImg2ImgPipeline
    prompt: str,
    init_image: Image.Image,
    negative_prompt: Optional[str] = None,
    strength: float = 0.7,
    num_inference_steps: int = 40,
    guidance_scale: float = 7.5,
    seed: Optional[int] = None,
    control_image: Optional[Image.Image] = None,
    controlnet_scale: float = 0.5,
):
    """Generates an image from an initial image using an SDXL pipeline, with optional ControlNet."""
    if seed is None:
        seed = random.randint(0, 2**32 - 1)
    generator = torch.Generator(device="cuda").manual_seed(seed)
    
    use_controlnet = control_image is not None

    output = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=init_image, # This is the init_image for this pipeline
        control_image=control_image, # This is the control_image
        strength=strength,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        controlnet_conditioning_scale=controlnet_scale if use_controlnet else 0.0,
        generator=generator,
    )
    return output.images[0]

In [ ]:
from fastapi import FastAPI
import nest_asyncio
import uvicorn

app = FastAPI()
nest_asyncio.apply()  # allow running uvicorn in a notebook

In [ ]:
from fastapi import Form, UploadFile, File, HTTPException
from fastapi.responses import StreamingResponse, JSONResponse
from io import BytesIO
from PIL import Image

@app.post("/generate")
async def generate(
    prompt: str = Form(...),
    negative_prompt: str = Form(None),
    seed: int = Form(None),
    num_inference_steps: int = Form(40),
    guidance_scale: float = Form(7.5),
    
    # Txt2Img params
    height: int = Form(1024),
    width: int = Form(1024),

    # Img2Img params
    strength: float = Form(0.7),
    init_image: UploadFile = File(None),  # If present, triggers Img2Img

    # ControlNet params
    control_image: UploadFile = File(None),
    controlnet_scale: float = Form(0.5),
):
    """Unified endpoint for SDXL generation. Switches between t2i and i2i based on `init_image`."""
    
    # --- Load ControlNet image if provided ---
    controlnet_img = None
    if control_image:
        try:
            contents = await control_image.read()
            controlnet_img = Image.open(BytesIO(contents)).convert("RGB")
        except Exception as e:
            return JSONResponse(status_code=400, content={"error": f"Failed to read control_image: {e}"})
    
    try:
        # --- Img2Img Workflow ---
        if init_image is not None:
            if not USE_IMG2IMG:
                raise RuntimeError("Received init_image, but notebook is in Txt2Img mode. Set USE_IMG2IMG=True.")
            
            init_img = Image.open(BytesIO(await init_image.read())).convert("RGB")
            generated_image = generate_img2img_xl(
                pipe=pipe,
                prompt=prompt, 
                init_image=init_img,
                negative_prompt=negative_prompt,
                strength=strength,
                num_inference_steps=num_inference_steps,
                guidance_scale=guidance_scale,
                seed=seed,
                control_image=controlnet_img,
                controlnet_scale=controlnet_scale
            )
        
        # --- Txt2Img Workflow ---
        else:
            if USE_IMG2IMG:
                raise RuntimeError("Did not receive init_image, but notebook is in Img2Img mode. Set USE_IMG2IMG=False.")
            
            generated_image = generate_txt2img_xl(
                pipe=pipe,
                prompt=prompt,
                negative_prompt=negative_prompt,
                height=height,
                width=width,
                num_inference_steps=num_inference_steps,
                guidance_scale=guidance_scale,
                seed=seed,
                control_image=controlnet_img,
                controlnet_scale=controlnet_scale
            )
    except Exception as e:
        return JSONResponse(status_code=500, content={"error": str(e)})

    # --- Return PNG stream ---
    buffer = BytesIO()
    generated_image.save(buffer, format="PNG")
    buffer.seek(0)
    return StreamingResponse(buffer, media_type="image/png")

In [ ]:
# Download zrok v1.1.3 (latest)
!wget https://github.com/openziti/zrok/releases/download/v1.1.3/zrok_1.1.3_linux_amd64.tar.gz
!tar -xzf zrok_1.1.3_linux_amd64.tar.gz
!chmod +x zrok

In [ ]:
# Enable (automatic migration from 0.4)
!./zrok enable --headless "$zrok_token"

In [ ]:
import uvicorn
import threading

def run_uvicorn():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

# Start in background thread
threading.Thread(target=run_uvicorn, daemon=True).start()

In [ ]:
import subprocess
import time

def start_zrok_tunnel(port=8000):
    # Start the tunnel
    process = subprocess.Popen([
        "./zrok", "share", "public", f"localhost:{port}", "--headless"
    ], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

    # Give it a moment to start
    time.sleep(3)

    # Check agent status to get the URL
    status_process = subprocess.run([
        "./zrok", "agent", "status"
    ], capture_output=True, text=True)

    print("Agent Status:")
    print(status_process.stdout)

    return process

# Start the tunnel
tunnel_process = start_zrok_tunnel(8000)
print("Zrok tunnel started! Check the agent status above for your public URL.")

In [ ]:
!./zrok overview

In [ ]:
import time

print("Server and zrok tunnel are running. Keeping the notebook alive...")

try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("Shutting down.")